# Active learning для сегментації

## Part 2. Uncertainty на синтетичних прогнозах

Перевіряємо margin uncertainty та predictive entropy без датасету, encoder і навчання. Коли з'явиться segmentation head, замінимо синтетичні logits його прогнозами. Функції не отримують GT masks або Oracle.

Логіка margin uncertainty відповідає [лекційному notebook, розділ 5](https://github.com/AI220-Computer-Vision/kse_advanced_computer_vision/blob/main/Module_2/Lecture_2/active_learning_moons.ipynb). Тут обчислюємо її для кожного пікселя у PyTorch, а потім усереднюємо по зображенню.

Цей блок працює на CPU. Потрібен лише PyTorch; завантажень, GPU-запусків та зовнішнього логування немає.

In [1]:
import math
import sys
from collections.abc import Iterable
from pathlib import Path
from typing import Literal

import torch
from torch import Tensor

# У Colab вкажи шлях до нашої локально зміненої версії source.
SOURCE_DIR = Path("src/source").resolve()
if not (SOURCE_DIR / "alseg").is_dir():
    raise FileNotFoundError("Вкажи SOURCE_DIR із каталогом alseg")
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

NUM_CLASSES = 21
IGNORE_INDEX = 255
SEED = 42
SIGMA_FLOOR = 1e-6
ECE_BINS = 5
TEMPERATURE_GRID = (0.25, 0.5, 1.0, 2.0, 4.0, 8.0)

### Від logits до pixel uncertainty

Вхід `logits` має форму `[B, C, H, W]` і dtype `float32`. Softmax застосовуємо по класах, тобто `dim=1`. У VOC `C=21`, включно з фоном. Вихід має форму `[B, H, W]`.

- **Margin uncertainty:** $U=1-(p_{(1)}-p_{(2)})$, де $p_{(1)}$ та $p_{(2)}$ — дві найбільші ймовірності. Близькі значення дають високу uncertainty.
- **Normalized predictive entropy:** $U=-\sum_{c=1}^{C}p_c\log p_c / \log C$. Враховує весь розподіл; рівномірний розподіл дає 1.

Обидва scores лежать у `[0, 1]`, але однакове число не означає однаковий зміст. Якщо два класи мають імовірності приблизно 0.5, margin uncertainty близька до 1, а entropy при 21 класі близька до $\log 2 / \log 21 \approx 0.228$.

`torch.special.entr` обчислює $-p\log p$ з визначеним значенням 0 при `p=0`. Температуру поки не підбираємо: використовуємо початкові logits, тобто `tau=1`. Жоден із цих scores сам по собі не розділяє epistemic та aleatoric uncertainty.

In [2]:
@torch.no_grad()
def pixel_uncertainty(
    logits: Tensor, method: Literal["margin", "entropy"]
) -> Tensor:
    """Перетворює raw logits [B, C, H, W] на uncertainty [B, H, W]."""
    if logits.ndim != 4 or min(logits.shape) < 1 or logits.shape[1] < 2:
        raise ValueError("Очікуємо непорожні logits [B, C, H, W], C >= 2")
    if logits.dtype != torch.float32 or not torch.isfinite(logits).all():
        raise ValueError("logits повинні мати dtype float32 без NaN та infinity")
    if method not in ("margin", "entropy"):
        raise ValueError("method має бути 'margin' або 'entropy'")

    probabilities = logits.softmax(dim=1)
    if method == "margin":
        top_two = probabilities.topk(k=2, dim=1).values
        uncertainty = 1.0 - (top_two[:, 0] - top_two[:, 1])
    else:
        uncertainty = torch.special.entr(probabilities).sum(dim=1)
        uncertainty = uncertainty / math.log(logits.shape[1])

    # Округлення float32 може дати значення трохи за теоретичними межами.
    return uncertainty.clamp(0.0, 1.0)

### Один score на зображення

Початкова агрегація — середня uncertainty по геометрично валідних пікселях. Ділимо на їхню кількість **окремо для кожного зображення**, а не на розмір усього batch.

`valid_pixels: bool [B, H, W]` описує лише геометрію, наприклад відсутність padding. Не можна будувати її з `GT != 255` для нерозміченого pool: це читання прихованої розмітки. Якщо padding немає, маска складається з `True`.

Недолік середнього: великий упевнений фон може зменшувати score зображення з малим невизначеним об'єктом. Поки фіксуємо цю агрегацію, а не підбираємо її за official val.

Функція зберігає порядок batch. Для реальних прогнозів передаватимемо `(batch.ids, image_scores)` у наявний `assemble_scores`, щоб відновити порядок pool IDs.

In [3]:
@torch.no_grad()
def mean_image_uncertainty(
    pixel_scores: Tensor, valid_pixels: Tensor
) -> Tensor:
    """Усереднює [B, H, W] до [B], виключаючи лише geometry padding."""
    if pixel_scores.ndim != 3 or min(pixel_scores.shape) < 1:
        raise ValueError("Очікуємо непорожні pixel_scores [B, H, W]")
    if pixel_scores.dtype != torch.float32 or not torch.isfinite(pixel_scores).all():
        raise ValueError("pixel_scores повинні мати dtype float32 без NaN та infinity")
    if ((pixel_scores < 0) | (pixel_scores > 1)).any():
        raise ValueError("pixel_scores повинні лежати в [0, 1]")
    if valid_pixels.dtype != torch.bool or valid_pixels.shape != pixel_scores.shape:
        raise ValueError("valid_pixels повинна бути bool-маскою тієї самої форми")
    if valid_pixels.device != pixel_scores.device:
        raise ValueError("Scores і geometry mask мають бути на одному device")

    counts = valid_pixels.sum(dim=(1, 2))
    if (counts == 0).any():
        raise ValueError("Кожне зображення повинно мати хоча б один валідний піксель")
    totals = pixel_scores.masked_fill(~valid_pixels, 0.0).sum(dim=(1, 2))
    return totals / counts

### Три контрольовані прогнози

Створимо три зображення по 2×3 пікселі. Усередині кожного прогноз однаковий у всіх пікселях, щоб легко перевірити відповідь. Це штучні logits, а не результати навченої моделі.

1. Один клас значно переважає: обидва scores близькі до 0.
2. Усі 21 класи рівноймовірні: обидва scores дорівнюють 1.
3. Два класи однаково правдоподібні: margin близька до 1, entropy близька до 0.228.

In [4]:
example_logits = torch.full((3, NUM_CLASSES, 2, 3), -12.0)
example_logits[0, 0] = 12.0
example_logits[1] = 0.0
example_logits[2, :2] = 12.0
example_valid_pixels = torch.ones((3, 2, 3), dtype=torch.bool)

example_scores = {}
for method in ("margin", "entropy"):
    pixel_scores = pixel_uncertainty(example_logits, method)
    example_scores[method] = mean_image_uncertainty(
        pixel_scores, example_valid_pixels
    )

print(f"{'Синтетичний прогноз':<28} {'margin':>10} {'entropy':>10}")
names = ("Один упевнений клас", "Рівномірний розподіл", "Два рівноймовірні класи")
for index, name in enumerate(names):
    print(
        f"{name:<28} {example_scores['margin'][index].item():10.6f}"
        f" {example_scores['entropy'][index].item():10.6f}"
    )

Синтетичний прогноз              margin    entropy
Один упевнений клас            0.000000   0.000000
Рівномірний розподіл           1.000000   1.000000
Два рівноймовірні класи        1.000000   0.227670


### Uncertainty top-B

Після агрегації маємо `scores: float32 [N]`. Індекс `scores[i]` — постійний pool ID, а не позиція зображення в поточному batch. Вибираємо `n_select` **нових** IDs лише з `unlabeled_ids`. Уже розмічені зображення не беруть участі, навіть якщо мають більшу uncertainty.

Порядок вибору: score за спаданням, а при однакових scores — менший ID. Для цього спочатку впорядковуємо IDs, потім застосовуємо `argsort(..., descending=True, stable=True)`. Звичайний `topk` не гарантує потрібного порядку при нічиїй.

`n_select=0` повертає порожній список; бюджет більший за `|U|` є помилкою. Функція не змінює вхідні дані та не звертається до Oracle. При cold start, коли навченого head ще немає, runner використає спільний Random fallback, а не вигадані uncertainty scores.

Нижче — шість синтетичних image scores. ID `3` уже labeled, тому score `1.0` не робить його кандидатом. IDs `1` і `4` мають однаковий score `0.95`; спочатку вибираємо `1`.

In [5]:
def uncertainty_top_b(
    scores: Tensor, unlabeled_ids: list[int], n_select: int
) -> list[int]:
    """Вибирає нові pool IDs за спаданням score; нічию вирішує менший ID."""
    if scores.ndim != 1 or scores.dtype != torch.float32:
        raise ValueError("Очікуємо scores: float32 [N]")
    if not torch.isfinite(scores).all() or (scores < 0).any():
        raise ValueError("Scores повинні бути скінченними та невід'ємними")
    if type(n_select) is not int or not 0 <= n_select <= len(unlabeled_ids):
        raise ValueError("n_select має бути цілим числом від 0 до розміру U")
    if any(type(i) is not int or not 0 <= i < len(scores) for i in unlabeled_ids):
        raise ValueError("Кандидати мають бути цілими pool IDs у межах scores")
    if len(set(unlabeled_ids)) != len(unlabeled_ids):
        raise ValueError("Кандидати не повинні містити повторних IDs")

    candidates = torch.tensor(sorted(unlabeled_ids), dtype=torch.long, device=scores.device)
    order = torch.argsort(scores[candidates], descending=True, stable=True)
    return candidates[order[:n_select]].tolist()

In [6]:
pool_scores = torch.tensor([0.10, 0.95, 0.80, 1.00, 0.95, 0.60])
unlabeled_ids = [0, 1, 2, 4, 5]  # ID 3 уже labeled.
selected_ids = uncertainty_top_b(pool_scores, unlabeled_ids, n_select=3)

print(f"Вибрані IDs у порядку пріоритету: {selected_ids}")
print("Їхні scores:", [round(pool_scores[i].item(), 2) for i in selected_ids])

Вибрані IDs у порядку пріоритету: [1, 4, 2]
Їхні scores: [0.95, 0.95, 0.8]


## Part 3. Gaussian kernel і MaxHerding

MaxHerding вибирає представників pool за схожістю embeddings. Прогнози head та мітки класів йому не потрібні, тому він працює і при cold start, коли `L` порожнє.

### Відстані та ширина kernel

Для реального запуску використовуємо L2-normalized image embeddings `[N, D]` від encoder. Рядки зберігають постійні pool IDs. Обчислюємо евклідові відстані один раз, потім:

$$\sigma = \operatorname{median}_{i<j}\|h_i-h_j\|_2, \qquad K_{i,j}=\exp\left(-\frac{\|h_i-h_j\|_2^2}{\sigma^2}\right).$$

Діагональ не бере участі в median, кожну пару враховуємо один раз. Для парної кількості відстаней беремо середнє двох центральних значень. Якщо median нульова через дублікати, застосовуємо floor `1e-6`. Для оцінки median потрібно хоча б два зображення.

Це label-free правило MaxHerding, а не adaptive sigma для UHerding. Його обчислимо на всьому train pool і залишимо незмінним між раундами за незмінних embeddings. Official val не використовуємо.

`cdist` рахує відстані без створення tensor `[N, N, D]`. Прямий режим обчислення зберігає нульові відстані між однаковими векторами без похибки від віднімання близьких квадратів норм. Повна kernel matrix для 1464 зображень у float32 займає приблизно 8.18 MiB; проміжні tensors потребують додаткової пам'яті.

In [7]:
@torch.no_grad()
def pairwise_distances(embeddings: Tensor) -> Tensor:
    """Евклідові відстані [N, N]; порядок рядків відповідає pool IDs."""
    if embeddings.ndim != 2 or min(embeddings.shape) == 0:
        raise ValueError("Очікуємо непорожні embeddings [N, D]")
    if embeddings.dtype != torch.float32 or not torch.isfinite(embeddings).all():
        raise ValueError("Embeddings мають бути скінченними float32")
    return torch.cdist(
        embeddings, embeddings, p=2, compute_mode="donot_use_mm_for_euclid_dist"
    )


@torch.no_grad()
def median_kernel_sigma(distances: Tensor) -> float:
    """Label-free sigma з матриці евклідових відстаней train pool."""
    if distances.shape[0] < 2:
        raise ValueError("Для median-відстані потрібно хоча б два зображення")
    rows, columns = torch.triu_indices(
        len(distances), len(distances), offset=1, device=distances.device
    )
    median = torch.quantile(distances[rows, columns], 0.5).item()
    return max(median, SIGMA_FLOOR)


@torch.no_grad()
def gaussian_kernel(distances: Tensor, sigma: float) -> Tensor:
    """Gaussian similarity [N, N] з евклідових відстаней, без множника 2."""
    if isinstance(sigma, bool) or not math.isfinite(sigma) or sigma <= 0:
        raise ValueError("sigma має бути скінченним додатним числом")
    return torch.exp(-(distances / sigma).square())

### Greedy-вибір за приростом покриття

Дотримуємося [лекційного notebook, розділ 4](https://github.com/AI220-Computer-Vision/kse_advanced_computer_vision/blob/main/Module_2/Lecture_2/active_learning_moons.ipynb). У `K` рядки — targets, стовпці — кандидати. Набір targets `U₀` фіксуємо на початку batch.

$$m_i=\max_{l\in L\cup S}K_{i,l},\qquad \Delta(c)=\sum_{i\in U_0}\max(K_{i,c}-m_i,0).$$

1. Якщо `L` порожнє, початкове покриття `m` нульове. Інакше враховуємо найкращу схожість із уже labeled зображеннями.
2. Вибираємо доступний ID із найбільшим gain. При нічиїй беремо найменший ID.
3. Оновлюємо `m = maximum(m, K[:, chosen])` і забороняємо повторний вибір цього кандидата.
4. Перераховуємо gains перед наступним вибором. Target row вибраної точки залишається в objective.

Ділення суми на незмінне `|U₀|` не змінює вибраний ID, тому в коді використовуємо суму. Навіть за нульових gains добираємо рівно `n_select` нових IDs за правилом нічиєї.

Функція отримує kernel від `gaussian_kernel`, а не embeddings, щоб не обчислювати матрицю повторно між раундами. `U` визначаємо як усі pool IDs, яких немає в `L`; маски та Oracle не передаємо.

In [8]:
@torch.no_grad()
def greedy_kernel_selection(
    kernel: Tensor,
    labeled_ids: list[int],
    n_select: int,
    weights: Tensor | None = None,
) -> list[int]:
    """Спільний greedy-цикл; weights [N] зважують targets, тобто rows."""
    if kernel.ndim != 2 or kernel.shape[0] != kernel.shape[1]:
        raise ValueError("Очікуємо квадратну kernel matrix [N, N]")
    n_pool = len(kernel)
    if any(type(i) is not int or not 0 <= i < n_pool for i in labeled_ids):
        raise ValueError("L має містити цілі IDs у межах pool")
    labeled = set(labeled_ids)
    if len(labeled) != len(labeled_ids):
        raise ValueError("L не повинно містити повторних IDs")
    unlabeled_ids = [i for i in range(n_pool) if i not in labeled]
    if type(n_select) is not int or not 0 <= n_select <= len(unlabeled_ids):
        raise ValueError("n_select має бути цілим числом від 0 до розміру U")
    if weights is not None:
        if weights.shape != (n_pool,) or weights.dtype != torch.float32:
            raise ValueError("Очікуємо weights: float32 [N] у порядку pool IDs")
        if weights.device != kernel.device or not torch.isfinite(weights).all():
            raise ValueError("Weights мають бути скінченними та на device kernel")
        if (weights < 0).any():
            raise ValueError("Weights не можуть бути від'ємними")
    if n_select == 0:
        return []

    targets = kernel[unlabeled_ids]  # [|U₀|, N], незмінні rows протягом batch.
    coverage = (
        targets[:, labeled_ids].amax(dim=1)
        if labeled_ids else kernel.new_zeros(len(unlabeled_ids))
    )
    target_weights = (
        kernel.new_ones(len(unlabeled_ids))
        if weights is None else weights[unlabeled_ids]
    )
    scale = target_weights.max()
    if scale > 0:
        target_weights = target_weights / scale  # Спільний множник не змінює argmax.

    candidates = targets[:, unlabeled_ids]  # Columns у порядку зростання pool ID.
    available = torch.ones(len(unlabeled_ids), dtype=torch.bool, device=kernel.device)
    selected_ids = []
    for _ in range(n_select):
        improvements = (candidates - coverage[:, None]).clamp_min_(0)
        gains = (improvements * target_weights[:, None]).sum(dim=0)
        gains.masked_fill_(~available, -torch.inf)
        position = gains.argmax().item()
        selected_ids.append(unlabeled_ids[position])
        coverage = torch.maximum(coverage, candidates[:, position])
        available[position] = False
    return selected_ids


def maxherding(kernel: Tensor, labeled_ids: list[int], n_select: int) -> list[int]:
    """MaxHerding використовує однакові ваги всіх targets."""
    return greedy_kernel_selection(kernel, labeled_ids, n_select)

### Чому не можна один раз взяти top-B за початковими gains

Візьмемо три групи з двома однаковими normalized embeddings у кожній: IDs `[0, 1]`, `[2, 3]`, `[4, 5]`. Дублікати тут навмисні, щоб показати надлишковий відбір.

За початковими gains найкращі IDs `2` та `3`, але вони представляють одну групу. MaxHerding спочатку вибирає `2`, оновлює покриття, а другим бере `0` з іншої групи. Якщо `2` уже labeled, його покриття враховується ще до першого нового вибору.

In [9]:
toy_embeddings = torch.tensor([
    [1.0, 0.0], [1.0, 0.0],
    [0.0, 1.0], [0.0, 1.0],
    [-1.0, 0.0], [-1.0, 0.0],
])
toy_distances = pairwise_distances(toy_embeddings)
toy_sigma = median_kernel_sigma(toy_distances)
toy_kernel = gaussian_kernel(toy_distances, toy_sigma)

one_shot_ids = torch.argsort(toy_kernel.sum(dim=0), descending=True, stable=True)[:2]
print(f"Median sigma: {toy_sigma:.6f}")
print("Top-2 початкових gains:", one_shot_ids.tolist())
print("MaxHerding з порожнім L:", maxherding(toy_kernel, labeled_ids=[], n_select=2))
print("Новий ID, якщо L = [2]:", maxherding(toy_kernel, labeled_ids=[2], n_select=1))

Median sigma: 1.414214
Top-2 початкових gains: [2, 3]
MaxHerding з порожнім L: [2, 0]
Новий ID, якщо L = [2]: [0]


### UHerding: uncertainty зважує targets

Використовуємо той самий greedy-цикл, але внесок кожного target залежить від його uncertainty:

$$\Delta_U(c)=\sum_{i\in U_0}u_i\max(K_{i,c}-m_i,0).$$

Вага `u_i` належить **рядку**, який кандидат допомагає покрити, а не самому кандидату. Точка з невеликою власною uncertainty теж може бути корисною, якщо представляє багато невизначених сусідів.

`uncertainty: float32 [N]` містить scores у порядку всього pool. Усередині відбору використовуємо лише ваги `U₀`; зміна ваг уже labeled точок не впливає на результат. Ділимо додатні target weights на їхній максимум: спільний множник не змінює argmax, а однакові додатні ваги стають точно одиницями. Якщо всі ваги нульові, всі gains нульові, тому добираємо найменші доступні IDs.

### Adaptive sigma за labeled embeddings

За [лекцією, розділ 5](https://github.com/AI220-Computer-Vision/kse_advanced_computer_vision/blob/main/Module_2/Lecture_2/active_learning_moons.ipynb):

$$\sigma^*=\min_{i,j\in L,\ i\ne j}\|h_i-h_j\|_2.$$

Беремо мінімум лише між різними labeled IDs, без діагоналі. При однакових embeddings застосовуємо `SIGMA_FLOOR`. Якщо labeled зображень менше двох, використовуємо явно передану label-free median sigma всього pool. Це fallback ширини kernel, не заміна Random cold start для методу без навченої моделі.

Відстані кешуємо, але kernel перебудовуємо, коли між раундами змінюється `σ`. Після появи двох labeled точок ця minimum-distance sigma не зростає при розширенні `L` за незмінних embeddings. Усередині одного batch `σ` та uncertainty фіксовані.

У цьому блоці scores синтетичні. Для реальних раундів вони мають надходити з поточного segmentation head; ECE та підбір temperature додамо окремо.

In [10]:
@torch.no_grad()
def adaptive_kernel_sigma(
    distances: Tensor, labeled_ids: list[int], fallback_sigma: float
) -> float:
    """Мінімальна відстань між різними labeled IDs; дані лише з train pool."""
    if any(type(i) is not int or not 0 <= i < len(distances) for i in labeled_ids):
        raise ValueError("L має містити цілі IDs у межах pool")
    if len(set(labeled_ids)) != len(labeled_ids):
        raise ValueError("L не повинно містити повторних IDs")
    if isinstance(fallback_sigma, bool) or not math.isfinite(fallback_sigma) or fallback_sigma <= 0:
        raise ValueError("fallback_sigma має бути скінченним додатним числом")
    if len(labeled_ids) < 2:
        return float(fallback_sigma)

    labeled_distances = distances[labeled_ids][:, labeled_ids]
    rows, columns = torch.triu_indices(
        len(labeled_ids), len(labeled_ids), offset=1, device=distances.device
    )
    minimum = labeled_distances[rows, columns].min().item()
    return max(minimum, SIGMA_FLOOR)


def uherding(
    kernel: Tensor, labeled_ids: list[int], n_select: int, uncertainty: Tensor
) -> list[int]:
    """UHerding використовує uncertainty як ваги target rows."""
    if uncertainty is None:
        raise ValueError("UHerding потребує scores; cold start визначає runner")
    return greedy_kernel_selection(kernel, labeled_ids, n_select, weights=uncertainty)

### Приклад впливу uncertainty

У двох ще не покритих груп однакова геометрія відносно `L`. Ліва група має uncertainty `0.1`, права — `0.9`. MaxHerding вирішує нічию за ID, а UHerding віддає перевагу правій групі.

Для цього порівняння обидва методи отримують **той самий kernel**, щоб ізолювати вплив ваг. У повному benchmark MaxHerding матиме власну median sigma, а UHerding — adaptive sigma.

In [11]:
weighted_embeddings = torch.tensor([
    [1.0, 0.0], [1.0, 0.0],
    [0.0, 1.0], [0.0, -1.0],
    [-1.0, 0.0], [-1.0, 0.0],
])
weighted_labeled_ids = [2, 3]
weighted_scores = torch.tensor([0.1, 0.1, 0.0, 0.0, 0.9, 0.9])
weighted_distances = pairwise_distances(weighted_embeddings)
weighted_sigma = adaptive_kernel_sigma(
    weighted_distances, weighted_labeled_ids,
    fallback_sigma=median_kernel_sigma(weighted_distances),
)
weighted_kernel = gaussian_kernel(weighted_distances, weighted_sigma)

print(f"Adaptive sigma: {weighted_sigma:.6f}")
print("MaxHerding:", maxherding(weighted_kernel, weighted_labeled_ids, 1))
print("UHerding:", uherding(weighted_kernel, weighted_labeled_ids, 1, weighted_scores))

Adaptive sigma: 2.000000
MaxHerding: [0]
UHerding: [4]


### Дві sanity checks з умови

За однакових **додатних** uncertainty weights та однакового kernel UHerding має збігатися з MaxHerding. За дуже малого `σ` і різних embeddings kernel наближається до identity, тому UHerding має збігатися з uncertainty top-B. Для другої перевірки не використовуємо дублікати з попереднього прикладу.

In [12]:
constant_scores = torch.full((6,), 0.7)
assert (
    uherding(weighted_kernel, [2, 3], 2, constant_scores)
    == maxherding(weighted_kernel, [2, 3], 2)
)

local_kernel = gaussian_kernel(pairwise_distances(torch.eye(4)), sigma=1e-8)
local_scores = torch.tensor([0.1, 0.9, 0.5, 0.9])
assert (
    uherding(local_kernel, [0], 2, local_scores)
    == uncertainty_top_b(local_scores, [1, 2, 3], 2)
)
print("Однакові додатні weights → MaxHerding; мале sigma → uncertainty top-B: OK")

Однакові додатні weights → MaxHerding; мале sigma → uncertainty top-B: OK


### Temperature calibration без official val

Temperature scaling перетворює logits на $p_\tau=\operatorname{softmax}(z/\tau)$. Додатна `τ` змінює впевненість, але не порядок класів. Обираємо її за ECE, а не за mIoU на official val.

Використовуємо протокол зі спільного `SETUP_PLAN.md`:

1. Перед новим query поділяємо попереднє queried `L` на fit і calibration за зображеннями.
2. Допоміжний acquisition head навчаємо лише на fit subset. Підбираємо `τ` на його прогнозах calibration subset.
3. Цей самий допоміжний head із цією `τ` обчислює uncertainty pool. Не донавчаємо його на calibration masks.
4. Після query новий reporting head навчаємо на всьому розширеному `L` та оцінюємо на official val. Temperature на нього не переносимо.

Це відрізняється від попередньої лекційної схеми з перенесенням temperature. Тут калібруємо саме модель, яка обчислює acquisition scores.

Використовуємо спільну функцію `split_labeled_ids` із `alseg`: canonical IDs, fixed calibration seed, 20% labeled images і щонайменше 4. За бюджету 21 це 17 fit і 4 calibration images. Для малих тестових бюджетів holdout обмежується `|L|-1`, щоб залишити хоча б одне fit image. `calibration_fraction=0` вимикає holdout; порожній holdout або відсутність валідних calibration pixels дають `τ=1` із записаною причиною fallback.

In [13]:
from alseg.core.calibration import split_labeled_ids


def split_calibration_ids(
    labeled_ids: list[int],
    seed: int,
    calibration_fraction: float = 0.2,
    min_calibration_images: int = 4,
) -> tuple[list[int], list[int]]:
    """Використовує спільний image-level split із alseg, без другого shuffle."""
    if any(type(i) is not int or i < 0 for i in labeled_ids):
        raise ValueError("Очікуємо невід'ємні цілі labeled IDs")
    if len(set(labeled_ids)) != len(labeled_ids):
        raise ValueError("L не повинно містити повторних IDs")
    if not 0 <= calibration_fraction < 1:
        raise ValueError("calibration_fraction має лежати в [0, 1)")
    if type(min_calibration_images) is not int or min_calibration_images < 0:
        raise ValueError("min_calibration_images має бути невід'ємним цілим числом")
    if type(seed) is not int or not 0 <= seed < 2**63:
        raise ValueError("seed має бути цілим числом у [0, 2**63)")
    if not labeled_ids:
        return [], []
    return split_labeled_ids(
        labeled_ids, calibration_fraction, min_calibration_images, seed
    )

### Pixel-level ECE

Confidence пікселя — найбільша ймовірність класу. Розбиваємо `[0, 1]` на 5 equal-width bins, як у лекції. Межі bins лівозамкнені, останній включає `1.0`.

$$\mathrm{ECE}=\sum_b\frac{n_b}{N}\left|\mathrm{accuracy}_b-\mathrm{confidence}_b\right|.$$

У кожному bin зберігаємо кількість пікселів, суму confidence і кількість правильних прогнозів. Ці суми можна додавати між batches. ECE рахуємо **після** додавання всіх сум, бо середнє batch-wise ECE загалом дає інший результат. Накопичуємо статистику у float64.

Враховуємо лише `valid_pixels & (targets != 255)`. Тут GT masks дозволені, бо це вже queried calibration subset. На unlabeled pool таке фільтрування за GT заборонене.

Кожен валідний піксель має однакову вагу, фон також входить у ECE. Велика частка фону може приховувати погану calibration рідкісних класів. За відсутності валідних пікселів ECE невизначене, а не нульове.

In [14]:
@torch.no_grad()
def ece_bin_sums(
    logits: Tensor,
    targets: Tensor,
    valid_pixels: Tensor,
    temperature: float = 1.0,
    n_bins: int = ECE_BINS,
) -> Tensor:
    """Повертає [3, n_bins]: counts, confidence sums, correct counts."""
    if logits.ndim != 4 or min(logits.shape) < 1 or logits.shape[1] < 2:
        raise ValueError("Очікуємо непорожні logits [B, C, H, W], C >= 2")
    if targets.shape != logits.shape[:1] + logits.shape[2:] or valid_pixels.shape != targets.shape:
        raise ValueError("Targets і geometry mask повинні мати форму [B, H, W]")
    if logits.dtype != torch.float32 or targets.dtype != torch.int64 or valid_pixels.dtype != torch.bool:
        raise ValueError("Очікуємо float32 logits, int64 targets та bool geometry mask")
    if targets.device != logits.device or valid_pixels.device != logits.device:
        raise ValueError("Усі tensors мають бути на одному device")
    if not torch.isfinite(logits).all():
        raise ValueError("Logits мають бути скінченними")
    if isinstance(temperature, bool) or not math.isfinite(temperature) or temperature <= 0:
        raise ValueError("Temperature має бути скінченним додатним числом")
    if type(n_bins) is not int or n_bins < 1:
        raise ValueError("n_bins має бути додатним цілим числом")

    keep = valid_pixels & (targets != IGNORE_INDEX)
    labels = targets[keep]
    if ((labels < 0) | (labels >= logits.shape[1])).any():
        raise ValueError("Calibration targets містять невідомий клас")
    confidence, predicted = (logits / temperature).softmax(dim=1).max(dim=1)
    confidence = confidence[keep].double()
    correct = (predicted[keep] == labels).double()
    bin_ids = (confidence * n_bins).long().clamp_max(n_bins - 1)
    return torch.stack([
        torch.bincount(bin_ids, minlength=n_bins).double(),
        torch.bincount(bin_ids, weights=confidence, minlength=n_bins).double(),
        torch.bincount(bin_ids, weights=correct, minlength=n_bins).double(),
    ])


def expected_calibration_error(bin_sums: Tensor) -> float:
    """ECE зі спільних bin sums усіх calibration batches."""
    n_pixels = bin_sums[0].sum()
    if n_pixels <= 0:
        raise ValueError("ECE невизначене без валідних calibration pixels")
    # n_b * |mean(confidence) - mean(correct)| = |sum(confidence) - sum(correct)|.
    return ((bin_sums[1] - bin_sums[2]).abs().sum() / n_pixels).item()

### Пошук temperature

Беремо лекційну сітку `0.25, 0.5, 1, 2, 4, 8`, зафіксовану до експериментів. Для кожної `τ` накопичуємо bin sums на всьому calibration subset і вибираємо найменше ECE. При точній нічиїй перемагає перше значення в сітці.

`fit_temperature` приймає iterator із tuples `(logits, targets, valid_pixels)`. Проходимо його один раз: inference не потрібно повторювати для кожної temperature, і всі calibration logits не треба тримати в RAM. Зберігаємо лише статистику bins та поточний batch.

Порожній iterator або відсутність валідних пікселів дають явний fallback `(1.0, {})`. Порожній словник означає, що ECE оцінити не вдалося. Для реальних даних tuples складатимемо з `CalibrationBatch` лише після перевірки походження його IDs.

In [15]:
@torch.no_grad()
def fit_temperature(
    calibration_batches: Iterable[tuple[Tensor, Tensor, Tensor]],
    temperatures: tuple[float, ...] = TEMPERATURE_GRID,
    n_bins: int = ECE_BINS,
) -> tuple[float, dict[float, float]]:
    """Повертає найкращу tau та ECE для кожної temperature на held-out L."""
    if not temperatures or any(
        isinstance(t, bool) or not math.isfinite(t) or t <= 0 for t in temperatures
    ):
        raise ValueError("Потрібна непорожня сітка скінченних додатних temperatures")
    if len(set(temperatures)) != len(temperatures):
        raise ValueError("Temperature grid не повинна містити повторів")
    if type(n_bins) is not int or n_bins < 1:
        raise ValueError("n_bins має бути додатним цілим числом")

    totals = torch.zeros(len(temperatures), 3, n_bins, dtype=torch.float64)
    for logits, targets, valid_pixels in calibration_batches:
        for index, temperature in enumerate(temperatures):
            totals[index] += ece_bin_sums(
                logits, targets, valid_pixels, temperature, n_bins
            ).cpu()

    if totals[0, 0].sum() == 0:
        return 1.0, {}
    errors = {
        float(temperature): expected_calibration_error(totals[index])
        for index, temperature in enumerate(temperatures)
    }
    best_temperature = min(errors, key=lambda temperature: errors[temperature])
    return best_temperature, errors

### Синтетичний приклад надмірної впевненості

Нижче окремо перевіряємо split для бюджету 21 і calibration на чотирьох штучних пікселях. Head не навчаємо. Для кожного пікселя модель дає класу `0` імовірність `0.99`, але правильними є лише три прогнози з чотирьох.

При `τ=1` ECE дорівнює приблизно `|0.99 - 0.75| = 0.24`. Більша temperature зменшить confidence, не змінюючи передбачений клас. Поліпшення ECE на цьому прикладі не означає поліпшення segmentation accuracy або гарантії на реальних даних.

In [16]:
fit_ids, calibration_ids = split_calibration_ids(list(range(21)), seed=SEED + 20_000)
print(f"Бюджет 21: training={len(fit_ids)}, calibration={len(calibration_ids)}")

calibration_probabilities = torch.full((NUM_CLASSES,), 0.01 / (NUM_CLASSES - 1))
calibration_probabilities[0] = 0.99
calibration_logits = calibration_probabilities.log()[None, :, None, None].expand(2, -1, 1, 2)
calibration_targets = torch.tensor([[[0, 0]], [[0, 1]]])
calibration_valid_pixels = torch.ones_like(calibration_targets, dtype=torch.bool)
calibration_batches = (
    (calibration_logits[i:i + 1], calibration_targets[i:i + 1], calibration_valid_pixels[i:i + 1])
    for i in range(2)
)
temperature, calibration_errors = fit_temperature(calibration_batches)
print(f"Обрана temperature: {temperature:g}")
print(f"ECE до calibration: {calibration_errors[1.0]:.6f}")
print(f"ECE після calibration: {calibration_errors[temperature]:.6f}")

# На реальному pool так само масштабуємо logits, але жодних GT masks не читаємо.
calibrated_pixel_scores = pixel_uncertainty(calibration_logits / temperature, method="margin")

Бюджет 21: training=17, calibration=4
Обрана temperature: 2
ECE до calibration: 0.239999
ECE після calibration: 0.060091


## Інтеграція з `alseg 0.9.0`

Базовий source взято з commit `424501366bc06499213279bd94c16d1a3f57e922` гілки `advanced/ass2`. Локально додаємо acquisition callback до спільного runner. Опублікований Kaggle source 0.9.0 ще не містить цього підключення: у Colab потрібно перенести саме наш каталог `src/source`, а не підмінити його незміненою Kaggle-копією.

Наші алгоритми, scoring і temperature fitting залишаються в notebook. `alseg.pipeline.run_experiment` відповідає за Oracle queries, новий reporting head, evaluation та artifacts. Другого AL training loop у notebook немає.

`SOURCE_DIR` задається на початку notebook. Потрібні залежності `src/source/pyproject.toml` із групами `train` та `encoder`, а також Jupyter і NumPy. Навіть у cache mode поточний source записує версії `transformers` і `timm` у metadata. Сам backbone для готового cache не створюється й weights не завантажуються. Автоматичного встановлення пакетів, завантаження VOC/features або навчання немає.

`NotebookSelector` реалізує `Selector.select(SelectionRequest)`. Він кешує відстані для одного незмінного набору embeddings і не отримує Oracle або evaluation split.

In [17]:
import json

import numpy as np

from alseg.config import EncoderConfig, EncoderName, ExperimentConfig
from alseg.core.calibration import CalibrationBatch
from alseg.core.predictions import LogitBatch
from alseg.core.scores import assemble_scores
from alseg.core.selection import AcquisitionMethod, SelectionRequest, validate_selection
from alseg.data.preprocessing import resize_mask
from alseg.interfaces.acquisition import Selector
from alseg.interfaces.data import ImageSource, Oracle
from alseg.interfaces.encoder import FeatureSource
from alseg.nn.extraction import iter_features
from alseg.nn.training import fit_head
from alseg.pipeline import prepare_labeled_features, run_experiment

In [18]:
class NotebookSelector(Selector):
    """Acquisition для одного frozen encoder і незмінного порядку pool IDs."""

    def __init__(self, embeddings: Tensor):
        self.embeddings = embeddings.detach().clone()
        self.distances = pairwise_distances(self.embeddings)
        self.median_sigma = (
            median_kernel_sigma(self.distances) if len(embeddings) > 1 else 1.0
        )

    def select(self, request: SelectionRequest) -> list[int]:
        if not torch.equal(request.embeddings, self.embeddings):
            raise ValueError("Embeddings не відповідають кешованим відстаням selector")
        if request.method == AcquisitionMethod.RANDOM:
            generator = torch.Generator().manual_seed(request.seed)
            order = torch.randperm(len(request.unlabeled_ids), generator=generator)
            selected = [request.unlabeled_ids[i] for i in order[:request.n_select].tolist()]
        elif request.method == AcquisitionMethod.UNCERTAINTY:
            selected = uncertainty_top_b(
                request.uncertainty, request.unlabeled_ids, request.n_select
            )
        else:
            kernel = gaussian_kernel(self.distances, request.sigma)
            if request.method == AcquisitionMethod.MAXHERDING:
                selected = maxherding(kernel, request.labeled_ids, request.n_select)
            else:
                selected = uherding(
                    kernel, request.labeled_ids, request.n_select, request.uncertainty
                )
        validate_selection(request, selected)
        return selected

### Передавання даних між компонентами

`FeatureSource` надає features або через frozen encoder, або з готового кешу колеги. У cache mode `iter_features` читає потрібні rows із диска, без RGB decoding і запуску backbone. Manifest і encoder identity перевіряє `CachedFeatures`. Для відбору окремо завантажуємо малий tensor image embeddings; усі patch features у RAM не копіюємо.

Pool inference отримує лише `ImageSource`, features і head. Oracle потрібен лише orchestration-функції для вже queried fit/calibration masks. Сам selector і image scoring не читають маски.

Для training і acquisition використовуємо квадратну роздільність `config["encoder"]["image_size"]`. Кадр масштабується цілком без padding, тому geometry mask складається з `True`. Official val оцінює функція колеги в початкових розмірах масок. Logits обробляємо батчами, не накопичуємо прогнози всього pool.

In [19]:
def extract_pool_embeddings(
    config: dict, pool: ImageSource, encoder: FeatureSource,
) -> Tensor:
    """Читає лише embeddings із кешу; fallback для image-only encoder."""
    directory = config["encoder"].get("features_dir")
    if directory is not None:
        # CachedFeatures перевіряє порядок image IDs відносно свого manifest.
        encoder.get_features(pool, [0])
        array = np.load(
            Path(directory) / "train" / "embeddings.npy",
            mmap_mode="r", allow_pickle=False,
        )
        return torch.from_numpy(np.array(array, copy=True))
    return torch.cat([
        features.embeddings
        for _, features in iter_features(
            pool, encoder, list(range(len(pool))), config["encoder"]["batch_size"]
        )
    ])


@torch.no_grad()
def iter_model_logits(
    head: torch.nn.Module, pool: ImageSource, encoder: FeatureSource,
    ids: list[int], image_size: int, batch_size: int, device: str,
) -> Iterable[LogitBatch]:
    """Потоковий inference без масок; після завершення head повертається на CPU."""
    head.to(device).eval()
    try:
        for batch_ids, features in iter_features(pool, encoder, ids, batch_size):
            logits = head(features.patch_features.to(device), (image_size, image_size)).cpu()
            valid_pixels = torch.ones_like(logits[:, 0], dtype=torch.bool)
            yield LogitBatch(batch_ids, logits, valid_pixels)
    finally:
        head.cpu()


def calibration_batches_from_logits(
    predictions: Iterable[LogitBatch], oracle: Oracle,
) -> Iterable[tuple[Tensor, Tensor, Tensor]]:
    """Лише прогнози на held-out queried IDs, визначених acquisition callback."""
    for batch in predictions:
        targets = torch.stack([
            resize_mask(oracle.get_mask(image_id), batch.logits.shape[-1])
            for image_id in batch.ids
        ])
        calibration = CalibrationBatch(batch, targets)
        yield calibration.predictions.logits, calibration.targets, batch.valid_pixels

### Знімки uncertainty для звіту

У benchmark callback додатково зберігає прогнози для двох найменш і двох найбільш невизначених зображень поточного `U`. Використовуємо той самий auxiliary head, `tau` та image-level score, що й для acquisition. RGB завантажимо лише під час побудови звіту. GT masks не читаємо.

Знімки `.npz` містять IDs, predictions і uncertainty maps. На Random cold start їх немає, бо навченого head ще не існує. Ця діагностика не змінює вибір IDs; для звичайного виклику `make_acquisition()` її вимкнено.

In [20]:
@torch.no_grad()
def save_uncertainty_snapshot(
    directory: Path, round_index: int, scores: Tensor, labeled_ids: list[int],
    head: torch.nn.Module, pool: ImageSource, encoder: FeatureSource,
    temperature: float, method: str, image_size: int, batch_size: int, device: str,
) -> str:
    """Зберігає прогнози для двох найменш і двох найбільш невизначених U images."""
    candidates = sorted(set(range(len(pool))) - set(labeled_ids))
    ordered = sorted(candidates, key=lambda image_id: (scores[image_id].item(), image_id))
    most = sorted(candidates, key=lambda image_id: (-scores[image_id].item(), image_id))[:2]
    ids = list(dict.fromkeys(ordered[:2] + most))
    predictions, uncertainty, saved_ids = [], [], []
    for batch in iter_model_logits(head, pool, encoder, ids, image_size, batch_size, device):
        saved_ids.extend(batch.ids)
        predictions.append(batch.logits.argmax(dim=1).numpy().astype(np.uint8))
        uncertainty.append(pixel_uncertainty(batch.logits / temperature, method).numpy())
    directory.mkdir(parents=True, exist_ok=True)
    path = directory / f"round-{round_index:02d}.npz"
    with path.open("xb") as stream:
        np.savez_compressed(
            stream, ids=np.asarray(saved_ids), scores=scores[saved_ids].numpy(),
            predictions=np.concatenate(predictions), uncertainty=np.concatenate(uncertainty),
            most_uncertain=np.asarray([image_id in most for image_id in saved_ids]),
            temperature=np.asarray(temperature),
        )
    return path.name

### Один acquisition callback, один спільний runner

`make_acquisition()` створює callback на один експеримент. Він отримує попереднє `L`, готує `SelectionRequest` і повертає нові IDs та acquisition metadata. Сам `Oracle.query` викликає лише спільний pipeline після перевірки IDs. Reporting head навчається з нуля на всьому новому `L`; його не передаємо в callback.

Uncertainty та UHerding мають спільний Random cold start. Перед наступними queries окремий acquisition head навчається на fit subset попереднього `L`. Він використовується і для temperature fitting на holdout, і для uncertainty всього pool.

Seed відбору: `42 + round_index`. Training seed: `10042`. Calibration seed: `20042`. Назва методу не впливає на seeds. Shared config зберігає всі ці значення та scoring settings.

`selection.json` містить вкладений `acquisition` із fit/calibration IDs, sigma, tau, ECE та fallback. Reporting `fit_ids` завжди покривають усе queried `L`. `acquisition_budget` описує стан до query, а `budget` у results — стан після query і reporting training. Для нового запуску потрібен новий output directory; resume не підтримується.

In [21]:
def make_acquisition(diagnostics_dir: Path | None = None):
    """Створює callback одного експерименту; query/training/eval належать pipeline."""
    selector = None
    pool_ids, encoder_identity = None, None

    def acquire(*, config, pool, oracle, encoder, round_index, budget):
        nonlocal selector, pool_ids, encoder_identity
        if selector is None:
            embeddings = extract_pool_embeddings(config, pool, encoder)
            if len(embeddings) != len(pool):
                raise ValueError("Embeddings мають покривати весь train pool")
            selector = NotebookSelector(embeddings)
            pool_ids = pool.image_ids
            encoder_identity = encoder.describe()
        elif pool.image_ids != pool_ids or encoder.describe() != encoder_identity:
            raise ValueError("Для іншого pool/encoder створи новий acquisition callback")

        method = AcquisitionMethod(config["acquisition"]["method"])
        protocol = config["protocol"]
        labeled_ids = list(oracle.labeled_ids)
        model_based = method in (AcquisitionMethod.UNCERTAINTY, AcquisitionMethod.UHERDING)
        selected_method = (
            AcquisitionMethod(protocol["cold_start"])
            if model_based and not labeled_ids else method
        )
        if not labeled_ids and model_based and selected_method != AcquisitionMethod.RANDOM:
            raise ValueError("Спільний cold start uncertainty/UHerding має бути random")
        temperature, errors, scores, sigma = None, {}, None, None
        fit_ids, calibration_ids = [], []
        snapshot = None
        fallback = "cold_start_random" if model_based and not labeled_ids else None
        if model_based and labeled_ids:
            fit_ids, calibration_ids = split_calibration_ids(
                labeled_ids,
                config["seed"] + protocol["calibration_seed_offset"],
                protocol["calibration_fraction"], protocol["calibration_min_images"],
            )
            features, targets = prepare_labeled_features(config, pool, oracle, encoder, fit_ids)
            task = fit_head(
                features, targets, config["task"],
                config["seed"] + protocol["training_seed_offset"],
                config["device"], logger=False,
            )
            del features, targets
            image_size = config["encoder"]["image_size"]
            batch_size = config["encoder"]["batch_size"]
            temperature = 1.0
            try:
                if calibration_ids:
                    predictions = iter_model_logits(
                        task.head, pool, encoder, calibration_ids,
                        image_size, batch_size, config["device"],
                    )
                    temperature, errors = fit_temperature(
                        calibration_batches_from_logits(predictions, oracle),
                        tuple(config["acquisition"]["temperatures"]),
                        config["acquisition"]["ece_bins"],
                    )
                    if not errors:
                        fallback = "no_valid_calibration_pixels"
                else:
                    fallback = "no_calibration_images"
                # Саме цей head використано для temperature fitting вище.
                scores = assemble_scores(len(pool), [
                    (batch.ids, mean_image_uncertainty(
                        pixel_uncertainty(
                            batch.logits / temperature,
                            config["acquisition"]["uncertainty_method"],
                        ), batch.valid_pixels,
                    ))
                    for batch in iter_model_logits(
                        task.head, pool, encoder, list(range(len(pool))),
                        image_size, batch_size, config["device"],
                    )
                ])
                if diagnostics_dir is not None:
                    snapshot = save_uncertainty_snapshot(
                        diagnostics_dir, round_index, scores, labeled_ids, task.head,
                        pool, encoder, temperature,
                        config["acquisition"]["uncertainty_method"],
                        image_size, batch_size, config["device"],
                    )
            finally:
                del task
        if selected_method == AcquisitionMethod.MAXHERDING:
            sigma = selector.median_sigma
        elif selected_method == AcquisitionMethod.UHERDING:
            sigma = adaptive_kernel_sigma(selector.distances, labeled_ids, selector.median_sigma)
        request = SelectionRequest(
            method=selected_method, embeddings=selector.embeddings,
            labeled_ids=labeled_ids,
            unlabeled_ids=sorted(set(range(len(pool))) - set(labeled_ids)),
            n_select=budget - len(labeled_ids),
            seed=config["seed"] + protocol["selection_seed_offset"] + round_index,
            uncertainty=scores, sigma=sigma,
        )
        return selector.select(request), {
            "method": method.value, "selection_method": selected_method.value,
            "selection_seed": request.seed, "acquisition_budget": len(labeled_ids),
            "calibration_fit_ids": fit_ids, "calibration_ids": calibration_ids,
            "tau": temperature, "sigma": sigma, "ece_by_temperature": errors,
            "fallback": fallback, "uncertainty_snapshot": snapshot,
        }

    return acquire

### Налаштування запуску в Colab

Перенеси notebook і наш оновлений `src/source`, встанови залежності та вкажи шляхи нижче. Feature root має містити `extraction.json`, `manifest.json` і підкаталоги `dino_vitb16` та `ibot_vitb16`. VOC із official train/val masks потрібен навіть із готовими features.

Початковий config нижче — smoke mode із двома бюджетами та двома epochs, не benchmark. `RUN_EXPERIMENT=False` блокує запуск: Restart & Run All виконує лише приклади й визначає функції. Перед дозволеним GPU-запуском виставимо `device="cuda"` та окремо перевіримо доступні ресурси.

Для benchmark потрібні повні бюджети, `smoke=False`, погоджені epochs і новий output directory. Усі чотири методи запускаються через цей самий pipeline.

In [22]:
RUN_EXPERIMENT = False
VOC_ROOT = Path("data/VOCdevkit/VOC2012")
FEATURES_ROOT = Path("data/alseg-features")
ENCODER_NAME = EncoderName.DINO_VITB16  # Інший варіант: EncoderName.IBOT_VITB16.
METHOD = "uherding"  # random, uncertainty, maxherding, uherding.
RUN_DIR = Path("runs") / ENCODER_NAME.value / f"{METHOD}-smoke-01"

experiment = ExperimentConfig(
    seed=SEED,
    budgets=[21, 42],
    data={"voc_root": str(VOC_ROOT)},
    encoder=EncoderConfig(
        name=ENCODER_NAME, image_size=448,
        features_dir=FEATURES_ROOT / ENCODER_NAME.value,
    ),
    acquisition={"method": METHOD},
    tracking={"mode": "disabled"},
    device="cpu",
    smoke=True,
)

In [23]:
if RUN_EXPERIMENT:
    extraction = json.loads((FEATURES_ROOT / "extraction.json").read_text())
    if not extraction.get("complete") or not extraction.get("reusable"):
        raise ValueError("Потрібен завершений повний feature cache, не extraction smoke")
    results = run_experiment(
        experiment.model_dump(mode="json"), RUN_DIR,
        experiment.device, experiment.tracking,
        smoke=experiment.smoke, acquisition=make_acquisition(),
    )
    for row in results:
        print(f"budget={row['budget']}, mIoU={row['miou']:.4f}")
    print("Результати:", RUN_DIR / "results.csv")
else:
    print("Запуск вимкнено. Перевір шляхи й config перед RUN_EXPERIMENT=True.")

Запуск вимкнено. Перевір шляхи й config перед RUN_EXPERIMENT=True.


## Part 5. Повний benchmark і звіт

Фінальна сітка: DINO та iBOT, чотири методи, бюджети `21, 42, 84, 168, 336, 672`, seed `42`. Окремо навчаємо full-data upper bound для кожного encoder. Усі reporting heads навчаються з нуля за однією recipe. Один seed не дозволяє оцінити статистичну значущість різниці.

Блоки нижче розділені: запуск створює artifacts, аналіз лише читає їх. `RUN_BENCHMARK=False` і `RUN_POSTHOC=False` роблять Restart & Run All безпечним. GPU-запуск дозволяємо окремо після перевірки короткого реального експерименту.

Для таблиць і графіків додатково потрібні `pandas>=2.2` та `matplotlib>=3.9`. `BENCHMARK_ROOT` має бути окремим каталогом, бажано на змонтованому persistent storage. `/content` зникає після видалення Colab runtime. Готові запуски можна пропускати, але частково завершені не перезаписуємо: resume у source не реалізовано.

In [24]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import display
from PIL import Image

from alseg.core.pool import PoolManifest
from alseg.nn.encoders import VIT16_WEIGHTS

RUN_BENCHMARK = False
RUN_POSTHOC = False  # Читання GT masks дозволено лише після завершення всієї сітки.
BENCHMARK_ROOT = Path("runs/benchmark-seed42")
BENCHMARK_DEVICE = "cuda"
BENCHMARK_BUDGETS = (21, 42, 84, 168, 336, 672)
BENCHMARK_ENCODERS = (EncoderName.DINO_VITB16, EncoderName.IBOT_VITB16)
BENCHMARK_METHODS = ("random", "uncertainty", "maxherding", "uherding")
FULL_POOL_SIZE = 1464
CHART_THEME = "light"  # Явно вибрані light або dark, незалежно від теми Colab.
METHOD_LABELS = {
    "random": "Random", "uncertainty": "Uncertainty top-B",
    "maxherding": "MaxHerding", "uherding": "UHerding",
}
VOC_CLASSES = (
    "background", "aeroplane", "bicycle", "bird", "boat", "bottle", "bus", "car",
    "cat", "chair", "cow", "diningtable", "dog", "horse", "motorbike", "person",
    "pottedplant", "sheep", "sofa", "train", "tvmonitor",
)


def benchmark_config(
    encoder_name: EncoderName, method: str, full_data: bool = False,
) -> ExperimentConfig:
    """Одна recipe для всієї сітки; full-data відрізняється лише бюджетом."""
    return ExperimentConfig(
        seed=SEED, budgets=[FULL_POOL_SIZE] if full_data else list(BENCHMARK_BUDGETS),
        data={"voc_root": str(VOC_ROOT)},
        paths={"cache_dir": "cache/alseg", "runs_dir": str(BENCHMARK_ROOT)},
        encoder=EncoderConfig(
            name=encoder_name, image_size=448, batch_size=8,
            features_dir=FEATURES_ROOT / encoder_name.value,
        ),
        acquisition={"method": method, "uncertainty_method": "margin"},
        task={"head": "linear_1x1", "epochs": 20, "batch_size": 4,
              "learning_rate": 0.001, "weight_decay": 0.0},
        tracking={"mode": "disabled"}, device=BENCHMARK_DEVICE, smoke=False,
    )


def benchmark_runs(root: Path):
    for encoder_name in BENCHMARK_ENCODERS:
        for method in (*BENCHMARK_METHODS, "full_data"):
            full_data = method == "full_data"
            config = benchmark_config(encoder_name, "random" if full_data else method, full_data)
            yield root / encoder_name.value / method, config, method

In [25]:
def read_benchmark_run(
    run_dir: Path, expected: ExperimentConfig, method: str,
) -> tuple[pd.DataFrame, str]:
    """Відхиляє іншу recipe, smoke results і дублікати; відсутність не є нулем."""
    if not run_dir.exists():
        return pd.DataFrame(), "missing"
    config_path = run_dir / "config.json"
    if not config_path.is_file():
        return pd.DataFrame(), "partial"
    actual = json.loads(config_path.read_text())
    wanted = expected.model_dump(mode="json")
    for key in ("seed", "budgets", "protocol", "task", "acquisition", "smoke"):
        if actual[key] != wanted[key]:
            raise ValueError(f"Несумісний {key}: {run_dir}")
    for key in ("name", "image_size", "batch_size"):
        if actual["encoder"][key] != wanted["encoder"][key]:
            raise ValueError(f"Несумісний encoder.{key}: {run_dir}")
    path = run_dir / "results.csv"
    if not path.is_file():
        return pd.DataFrame(), "partial"
    data = pd.read_csv(path)
    required = {
        "encoder", "method", "budget", "round", "seed", "epochs", "smoke", "val_images",
        "miou", "pixel_accuracy", "mean_class_accuracy", "val_loss", "n_fit", "n_calibration",
        "acquisition_budget", "sigma_used", "tau_used",
        "selection_seconds", "feature_seconds", "train_seconds", "eval_seconds",
    }
    if not required <= set(data.columns):
        raise ValueError(f"Неповна схема results.csv: {run_dir}")
    budgets = data.budget.tolist()
    if budgets != expected.budgets[:len(data)] or data.budget.duplicated().any():
        raise ValueError(f"Неправильна послідовність бюджетів: {run_dir}")
    valid = (
        (data.encoder == expected.encoder.name.value) & (data.method == expected.acquisition.method.value)
        & (data.seed == expected.seed) & (data.epochs == expected.task.epochs)
        & data.smoke.eq(False) & (data.val_images == 1449)
        & (data.n_fit == data.budget) & (data.n_calibration == 0)
    )
    metrics = data[["miou", "pixel_accuracy", "mean_class_accuracy"]].to_numpy(dtype=float)
    if not valid.all() or not np.isfinite(metrics).all() or ((metrics < 0) | (metrics > 1)).any():
        raise ValueError(f"Непридатні benchmark results: {run_dir}")
    if not np.isfinite(data.val_loss.to_numpy(dtype=float)).all() or (data.val_loss < 0).any():
        raise ValueError(f"Непридатний validation loss: {run_dir}")
    for index, budget in enumerate(budgets):
        round_config = json.loads((run_dir / f"round-{index:02d}-budget-{budget}" / "config.json").read_text())
        identity = round_config["encoder_identity"]
        if (
            identity["name"] != expected.encoder.name.value
            or identity["image_size"] != expected.encoder.image_size
            or identity["weights_sha256"] != VIT16_WEIGHTS[expected.encoder.name]["sha256"]
            or round_config["runtime"]["precision"] != "32-true"
            or round_config["task"] != wanted["task"]
        ):
            raise ValueError(f"Інші backbone, precision або recipe: {run_dir}")
    data = data.assign(method=method, run_dir=str(run_dir))
    calibrated = data.method.isin(["uncertainty", "uherding"]) & (data.acquisition_budget > 0)
    data.loc[~calibrated, "tau_used"] = np.nan
    state_path = run_dir / "state.json"
    state = json.loads(state_path.read_text()) if state_path.exists() else {}
    complete = (
        budgets == expected.budgets and state.get("status") == "completed"
        and state.get("budget") == expected.budgets[-1]
    )
    return data, "complete" if complete else "partial"


def collect_benchmark(root: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    frames, statuses, manifest_hashes = [], [], set()
    for run_dir, config, method in benchmark_runs(root):
        frame, status = read_benchmark_run(run_dir, config, method)
        statuses.append({"encoder": config.encoder.name.value, "method": method,
                         "status": status, "rounds": len(frame)})
        if not frame.empty:
            saved = json.loads((run_dir / "manifest.json").read_text())
            manifest = PoolManifest(saved["pool_ids"], saved["val_ids"])
            if manifest.sha256 != saved["sha256"] or (len(manifest.pool_ids), len(manifest.val_ids)) != (1464, 1449):
                raise ValueError(f"Пошкоджений або неповний manifest: {run_dir}")
            manifest_hashes.add(manifest.sha256)
            frames.append(frame)
    if len(manifest_hashes) > 1:
        raise ValueError("Не можна об'єднувати результати з різних pool/val manifests")
    return (pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()), pd.DataFrame(statuses)


def benchmark_tables(data: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    baseline = data.loc[data.method == "random", ["encoder", "budget", "miou"]]
    detailed = data.merge(
        baseline.rename(columns={"miou": "random_miou"}),
        on=["encoder", "budget"], how="left", validate="many_to_one",
    )
    detailed["gain_over_random_pp"] = 100 * (detailed.miou - detailed.random_miou)
    summary = detailed.pivot(index=["encoder", "method"], columns="budget", values="miou")
    areas = {}
    for key, group in detailed.groupby(["encoder", "method"], sort=False):
        group = group.sort_values("budget")
        areas[key] = np.nan
        if group.budget.tolist() == list(BENCHMARK_BUDGETS):
            x, y = group.budget.to_numpy(), group.miou.to_numpy()
            areas[key] = float(np.sum(np.diff(x) * (y[:-1] + y[1:]) / 2) / (x[-1] - x[0]))
    summary["normalized_AUBC"] = pd.Series(areas)
    return detailed, summary

In [26]:
if RUN_BENCHMARK:
    extraction = json.loads((FEATURES_ROOT / "extraction.json").read_text())
    if not extraction.get("complete") or not extraction.get("reusable"):
        raise ValueError("Для benchmark потрібен повний reusable feature cache")
    if BENCHMARK_DEVICE == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("GPU не під'єднана. Benchmark не розпочато")
    pending_runs = []
    for run_dir, config, method in benchmark_runs(BENCHMARK_ROOT):
        _, status = read_benchmark_run(run_dir, config, method)
        if status == "partial":
            raise RuntimeError(f"Незавершений запуск не перезаписуємо: {run_dir}")
        if status == "missing":
            pending_runs.append((run_dir, config, method))
    for run_dir, config, method in pending_runs:
        print(f"Запуск {config.encoder.name.value}: {method}", flush=True)
        callback = None if method == "full_data" else make_acquisition(run_dir / "diagnostics")
        run_experiment(
            config.model_dump(mode="json"), run_dir, config.device, config.tracking,
            smoke=False, acquisition=callback,
        )
else:
    print("Повний benchmark вимкнено. Аналіз нижче не запускає навчання.")

Повний benchmark вимкнено. Аналіз нижче не запускає навчання.


### Як читати таблиці

`mIoU`, mean class accuracy та pixel accuracy зберігаємо в `[0, 1]`. На learning curves показуємо mIoU у відсотках. Gain over Random вимірюємо у процентних пунктах за однакових encoder, seed і budget.

Normalized AUBC визначаємо як площу під кусочно-лінійною кривою mIoU на **лінійній** осі budget, поділену на `672 − 21`. Log x-axis використовується лише для відображення learning curves. Не додаємо вигадану точку при нульовому бюджеті й не включаємо full-data upper bound у AUBC. Неповна траєкторія отримує `NaN`, а не оманливе значення за іншим інтервалом.

`acquisition_budget` у графіках `sigma` і `tau` означає розмір `L` **до** query. Порожні cold-start значення не замінюємо на `0` або `1`. Допоміжні метрики показують інші властивості сегментації, але не використовуються для tuning на official val.

### Як читати криві навчання

Train cross-entropy та train mIoU показуємо за epochs окремо для кожного encoder і бюджету. Кожна крива належить новому reporting head, який навчався на всьому поточному `L`. Auxiliary heads сюди не підмішуємо. Epochs на графіках і в `training-history.csv` нумеруємо від 1 до 20. Train loss усереднено за валідними пікселями під час epoch, а train mIoU обчислено зі спільної confusion matrix.

Validation loss показуємо за бюджетом після завершення training. Runner не зберігав validation loss для кожної epoch, тому такої кривої тут немає. Official val не впливає на fitting, early stopping чи вибір epoch. Менший train loss сам по собі не означає кращу якість на val. Усі криві читаємо з artifacts, без повторного training.


In [27]:
CHART_PALETTES = {
    "light": ("#2a78d6", "#eb6834", "#1baf7a", "#eda100"),
    "dark": ("#3987e5", "#d95926", "#199e70", "#c98500"),
}
CHART_SURFACES = {"light": "#fcfcfb", "dark": "#1a1a19"}
CHART_INK = {"light": "#0b0b0b", "dark": "#f0efec"}
CHART_GRID = {"light": "#e1e0d9", "dark": "#2c2c2a"}


def chart_style(theme: str) -> dict:
    """Кольори entity фіксовані; markers і line styles дублюють їхнє значення."""
    return {
        "figure.facecolor": CHART_SURFACES[theme], "axes.facecolor": CHART_SURFACES[theme],
        "text.color": CHART_INK[theme], "axes.labelcolor": CHART_INK[theme],
        "xtick.color": CHART_INK[theme], "ytick.color": CHART_INK[theme],
        "axes.edgecolor": CHART_GRID[theme], "grid.color": CHART_GRID[theme],
        "grid.linewidth": 0.7, "axes.spines.top": False, "axes.spines.right": False,
        "font.size": 10, "lines.linewidth": 2, "lines.markersize": 8,
        "savefig.facecolor": CHART_SURFACES[theme],
    }


def draw_methods(
    ax, data: pd.DataFrame, x: str, y: str, theme: str, scale: float = 1,
    markevery: int | None = None,
):
    for index, method in enumerate(BENCHMARK_METHODS):
        group = data.loc[data.method == method].dropna(subset=[x, y]).sort_values(x)
        if group.empty:
            continue
        ax.plot(
            group[x], group[y] * scale, label=METHOD_LABELS[method],
            color=CHART_PALETTES[theme][index], marker=("o", "s", "^", "D")[index],
            linestyle=("-", "--", "-.", ":")[index], markevery=markevery,
            markeredgecolor=CHART_SURFACES[theme], markeredgewidth=1.5,
        )
    ax.grid(axis="y")
    if ax.lines:
        ax.legend(frameon=False)


def plot_learning_curves(results: pd.DataFrame, theme: str = CHART_THEME):
    """Одна panel на encoder; full-data оцінка є горизонтальним орієнтиром."""
    with plt.rc_context(chart_style(theme)):
        fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), sharey=True, layout="constrained")
        for ax, encoder_name in zip(axes, BENCHMARK_ENCODERS):
            data = results.loc[results.encoder == encoder_name.value]
            draw_methods(ax, data.loc[data.method != "full_data"], "budget", "miou", theme, 100)
            upper = data.loc[data.method == "full_data", "miou"]
            if not upper.empty:
                ax.axhline(float(upper.iloc[0]) * 100, color=CHART_INK[theme],
                           linewidth=1, linestyle="--", label="Full-data upper bound")
                ax.legend(frameon=False)
            ax.set(xscale="log", xticks=BENCHMARK_BUDGETS, ylim=(0, 100),
                   title=encoder_name.value, xlabel="Кількість labeled images")
            ax.set_xticklabels(BENCHMARK_BUDGETS)
            ax.minorticks_off()
        axes[0].set_ylabel("Val mIoU, %")
        fig.suptitle("Learning curves, seed 42")
        return fig


def plot_adaptive_parameters(results: pd.DataFrame, theme: str = CHART_THEME):
    with plt.rc_context(chart_style(theme)):
        fig, axes = plt.subplots(2, 2, figsize=(12, 7), layout="constrained")
        for column, encoder_name in enumerate(BENCHMARK_ENCODERS):
            data = results.loc[results.encoder == encoder_name.value]
            for row, (metric, label) in enumerate((("sigma_used", "Sigma"), ("tau_used", "Tau"))):
                ax = axes[row, column]
                draw_methods(ax, data, "acquisition_budget", metric, theme)
                ax.set(title=encoder_name.value, xlabel="Labeled images до query", ylabel=label)
        return fig


def collect_training_history(results: pd.DataFrame) -> pd.DataFrame:
    # Читаємо лише reporting heads; auxiliary training сюди не підмішуємо.
    frames = []
    metrics = ["train/loss", "train/miou", "train/pixel_accuracy"]
    for row in results.itertuples(index=False):
        path = Path(row.run_dir) / f"round-{row.round:02d}-budget-{row.budget}" / "training.csv"
        history = pd.read_csv(path)
        if not {"epoch", *metrics} <= set(history.columns):
            raise ValueError(f"Неповна схема training.csv: {path}")
        values = history[metrics].to_numpy(dtype=float)
        if (
            history.epoch.tolist() != list(range(int(row.epochs)))
            or not np.isfinite(values).all()
            or (values[:, 0] < 0).any()
            or ((values[:, 1:] < 0) | (values[:, 1:] > 1)).any()
        ):
            raise ValueError(f"Неповна або пошкоджена історія training: {path}")
        frames.append(history[["epoch", *metrics]].assign(
            encoder=row.encoder, method=row.method, budget=row.budget, round=row.round,
            epoch=history.epoch + 1,
        ))
    columns = ["encoder", "method", "budget", "round", "epoch", *metrics]
    return pd.concat(frames, ignore_index=True)[columns] if frames else pd.DataFrame(columns=columns)


def plot_training_history(
    history: pd.DataFrame, encoder: str, metric: str, theme: str = CHART_THEME,
):
    labels = {"train/loss": "Train cross-entropy", "train/miou": "Train mIoU, %"}
    scale = 100 if metric == "train/miou" else 1
    data = history.loc[(history.encoder == encoder) & (history.method != "full_data")]
    with plt.rc_context(chart_style(theme)):
        fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=True, sharey=True, layout="constrained")
        for ax, budget in zip(axes.flat, BENCHMARK_BUDGETS):
            draw_methods(ax, data.loc[data.budget == budget], "epoch", metric, theme, scale, markevery=5)
            ax.set(title=f"Budget {budget}", xlabel="Epoch", ylabel=labels[metric],
                   xticks=[1, 5, 10, 15, 20], xlim=(1, 20))
            ax.set_ylim(bottom=0)
            if metric == "train/miou":
                ax.set_ylim(0, 100)
        fig.suptitle(f"{encoder}: reporting heads, seed 42")
        return fig


def plot_full_data_history(history: pd.DataFrame, theme: str = CHART_THEME):
    with plt.rc_context(chart_style(theme)):
        fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True, layout="constrained")
        for column, encoder in enumerate(BENCHMARK_ENCODERS):
            data = history.loc[(history.encoder == encoder.value) & (history.method == "full_data")].sort_values("epoch")
            for row, (metric, label, scale) in enumerate((
                ("train/loss", "Train cross-entropy", 1), ("train/miou", "Train mIoU, %", 100),
            )):
                ax = axes[row, column]
                ax.plot(data.epoch, data[metric] * scale, color=CHART_PALETTES[theme][0],
                        marker="o", markevery=5, markeredgecolor=CHART_SURFACES[theme])
                ax.set(title=encoder.value, xlabel="Epoch", ylabel=label,
                       xticks=[1, 5, 10, 15, 20], xlim=(1, 20))
                ax.set_ylim(bottom=0)
                if metric == "train/miou":
                    ax.set_ylim(0, 100)
                ax.grid(axis="y")
        fig.suptitle("Full-data reporting heads: 1464 images, seed 42")
        return fig


def plot_validation_loss(results: pd.DataFrame, theme: str = CHART_THEME):
    # Official val оцінено лише після останньої epoch кожного reporting head.
    with plt.rc_context(chart_style(theme)):
        fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), sharey=True, layout="constrained")
        for ax, encoder in zip(axes, BENCHMARK_ENCODERS):
            data = results.loc[results.encoder == encoder.value]
            draw_methods(ax, data.loc[data.method != "full_data"], "budget", "val_loss", theme)
            reference = data.loc[data.method == "full_data", "val_loss"]
            if not reference.empty:
                ax.axhline(float(reference.iloc[0]), color=CHART_INK[theme],
                           linewidth=1, linestyle="--", label="Full-data reference")
                ax.legend(frameon=False)
            ax.set(title=encoder.value, xlabel="Кількість labeled images", ylabel="Val cross-entropy",
                   xscale="log", xticks=BENCHMARK_BUDGETS)
            ax.set_xticklabels(BENCHMARK_BUDGETS)
            ax.set_ylim(bottom=0)
            ax.minorticks_off()
        fig.suptitle("Validation loss після навчання head, seed 42")
        return fig


In [28]:
benchmark_results, benchmark_status = collect_benchmark(BENCHMARK_ROOT)
display(benchmark_status)
if benchmark_results.empty:
    print("Benchmark results ще немає. Таблиці й графіки з'являться після реальних запусків.")
else:
    round_table, summary_table = benchmark_tables(benchmark_results)
    regimes = {21: "low", 42: "low", 84: "mid", 168: "mid", 336: "high", 672: "high"}
    active = round_table.loc[round_table.method != "full_data"].copy()
    regime_table = (
        active.assign(regime=active.budget.map(regimes))
        .groupby(["encoder", "regime", "method"]).miou.agg(["mean", "count"])
    )
    regime_table.loc[regime_table["count"] != 2, "mean"] = np.nan
    comparison_table = active.pivot(index=["encoder", "budget"], columns="method", values="miou")
    comparison_table = comparison_table.reindex(columns=BENCHMARK_METHODS)
    comparison_table["uherding_vs_better_baseline_pp"] = 100 * (
        comparison_table.uherding
        - comparison_table[["maxherding", "uncertainty"]].max(axis=1, skipna=False)
    )
    display(summary_table.round(4))
    display(round_table[["encoder", "method", "budget", "miou", "pixel_accuracy",
                         "mean_class_accuracy", "gain_over_random_pp"]].round(4))
    display(regime_table.round(4))
    display(comparison_table.round(4))
    timing_table = round_table[["encoder", "method", "budget", "selection_seconds",
                                "feature_seconds", "train_seconds", "eval_seconds"]]
    display(timing_table.round(2))
    report_dir = BENCHMARK_ROOT / "report"
    report_dir.mkdir(parents=True, exist_ok=True)
    training_history = collect_training_history(benchmark_results)
    training_history.to_csv(report_dir / "training-history.csv", index=False)
    round_table.to_csv(report_dir / "per-round.csv", index=False)
    summary_table.to_csv(report_dir / "summary.csv")
    regime_table.to_csv(report_dir / "regimes.csv")
    comparison_table.to_csv(report_dir / "uherding-comparison.csv")
    timing_table.to_csv(report_dir / "timings.csv", index=False)
    for name, plot in (("learning-curves", plot_learning_curves),
                       ("adaptive-parameters", plot_adaptive_parameters),
                       ("validation-loss", plot_validation_loss)):
        figure = plot(benchmark_results)
        figure.savefig(report_dir / f"{name}.png", dpi=150)
        plt.show()
        plt.close(figure)

    for encoder in BENCHMARK_ENCODERS:
        for metric, suffix in (("train/loss", "loss"), ("train/miou", "miou")):
            figure = plot_training_history(training_history, encoder.value, metric)
            figure.savefig(report_dir / f"training-{suffix}-{encoder.value}.png", dpi=150)
            plt.show()
            plt.close(figure)
    figure = plot_full_data_history(training_history)
    figure.savefig(report_dir / "full-data-training.png", dpi=150)
    plt.show()
    plt.close(figure)

encoder_rows = []
for encoder_name in BENCHMARK_ENCODERS:
    path = FEATURES_ROOT / encoder_name.value / "metadata.json"
    if path.is_file():
        metadata = json.loads(path.read_text())
        encoder_rows.append({
            "encoder": encoder_name.value,
            "parameters": metadata["encoder_identity"]["num_parameters"],
            "extraction_seconds": metadata["extraction_seconds"],
            "extraction_peak_cuda_gib": metadata["peak_cuda_bytes"] / 1024**3,
            "gpu": metadata["gpu"], "source": "metadata extraction колеги",
        })
if encoder_rows:
    encoder_table = pd.DataFrame(encoder_rows)
    display(encoder_table.round(3))
    if not benchmark_results.empty:
        encoder_table.to_csv(report_dir / "encoders.csv", index=False)

,encoder,method,status,rounds
0,dino_vitb16,random,missing,0
1,dino_vitb16,uncertainty,missing,0
2,dino_vitb16,maxherding,missing,0
3,dino_vitb16,uherding,missing,0
4,dino_vitb16,full_data,missing,0
5,ibot_vitb16,random,missing,0
6,ibot_vitb16,uncertainty,missing,0
7,ibot_vitb16,maxherding,missing,0
8,ibot_vitb16,uherding,missing,0
9,ibot_vitb16,full_data,missing,0


Benchmark results ще немає. Таблиці й графіки з'являться після реальних запусків.


### Відбір, annotation cost та uncertainty maps

Post-hoc аналіз виконуємо лише після завершення всіх 10 запусків. Він читає train masks для розподілу класів і cost proxy, але не передає їх у selection, calibration або training. Official val не використовується для вибору методу uncertainty чи параметрів.

Розподіл класів показуємо як частку зображень, де присутній клас; ці частки не повинні сумуватися до 100%. Додатково зберігаємо частки валідних пікселів. Cost proxy — кількість горизонтальних і вертикальних сусідніх пар пікселів із різними класами в початковій масці, без пар із `255`. Це довжина semantic boundaries у grid units, а не виміряний час анотування й не кількість instances.

Uncertainty maps використовують збережені прогнози auxiliary head до query. На всіх картах однакова шкала `[0, 1]`. Поруч показуємо RGB і межі передбачених класів без читання GT. Висока uncertainty не доводить окремо epistemic чи aleatoric походження.

`selection_seconds` включає auxiliary training, calibration, scoring та запис діагностики. `train_seconds` стосується лише reporting head. Пам'ять extraction з metadata колеги не є вимірюванням пам'яті training.

In [29]:
def mask_summary(mask: np.ndarray) -> tuple[np.ndarray, int]:
    """Post-hoc counts і semantic boundary length; void не утворює межу."""
    valid = mask != IGNORE_INDEX
    labels = mask[valid]
    if mask.ndim != 2 or ((labels < 0) | (labels >= NUM_CLASSES)).any():
        raise ValueError("Очікуємо VOC mask [H, W] із класами 0..20 або 255")
    counts = np.bincount(labels, minlength=NUM_CLASSES)
    horizontal = (mask[:, 1:] != mask[:, :-1]) & valid[:, 1:] & valid[:, :-1]
    vertical = (mask[1:] != mask[:-1]) & valid[1:] & valid[:-1]
    return counts, int(horizontal.sum() + vertical.sum())


def posthoc_tables(data: pd.DataFrame, voc_root: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Читає train labels лише для окремого post-hoc звіту, по одній масці."""
    from alseg.data.voc import load_manifest

    voc_manifest = load_manifest(voc_root)
    counts, boundaries = [], []
    for stem in voc_manifest.pool_ids:
        with Image.open(voc_root / "SegmentationClass" / f"{stem}.png") as image:
            class_counts, length = mask_summary(np.array(image))
        counts.append(class_counts)
        boundaries.append(length)
    counts, boundaries = np.asarray(counts), np.asarray(boundaries)
    distributions, costs = [], []
    for run_dir, group in data.groupby("run_dir", sort=False):
        directory = Path(run_dir)
        manifest = json.loads((directory / "manifest.json").read_text())
        if manifest["pool_ids"] != voc_manifest.pool_ids or manifest["val_ids"] != voc_manifest.val_ids:
            raise ValueError("VOC і experiment manifest не збігаються")
        encoder, method = group.iloc[0][["encoder", "method"]]
        selections = json.loads((directory / "selections.json").read_text())
        previous = set()
        for selection in selections:
            ids, new_ids = selection["ids"], selection["new_ids"]
            if (
                len(ids) != selection["budget"] or len(set(ids)) != len(ids)
                or any(type(i) is not int or not 0 <= i < len(counts) for i in ids)
                or len(set(new_ids)) != len(new_ids) or previous & set(new_ids)
                or previous | set(new_ids) != set(ids)
            ):
                raise ValueError(f"Непослідовні queried IDs: {directory}")
            previous = set(ids)
            selected = counts[ids]
            totals = selected.sum(axis=0)
            shares = totals / totals.sum() if totals.sum() else np.full(NUM_CLASSES, np.nan)
            distributions.append(pd.DataFrame({
                "encoder": encoder, "method": method, "budget": len(ids),
                "class": VOC_CLASSES, "image_presence_pct": (selected > 0).mean(axis=0) * 100,
                "pixel_share": shares,
            }))
            costs.append({"encoder": encoder, "method": method, "budget": len(ids),
                          "boundary_edges": int(boundaries[ids].sum())})
    totals = counts.sum(axis=0)
    for encoder in data.encoder.unique():
        distributions.append(pd.DataFrame({
            "encoder": encoder, "method": "pool", "budget": len(counts), "class": VOC_CLASSES,
            "image_presence_pct": (counts > 0).mean(axis=0) * 100,
            "pixel_share": totals / totals.sum() if totals.sum() else np.nan,
        }))
    cost_table = data.merge(pd.DataFrame(costs), on=["encoder", "method", "budget"],
                            validate="one_to_one")
    return cost_table, pd.concat(distributions, ignore_index=True)


def plot_annotation_cost(cost_table: pd.DataFrame, theme: str = CHART_THEME):
    with plt.rc_context(chart_style(theme)):
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True, layout="constrained")
        for ax, encoder_name in zip(axes, BENCHMARK_ENCODERS):
            data = cost_table.loc[cost_table.encoder == encoder_name.value]
            draw_methods(ax, data, "boundary_edges", "miou", theme, 100)
            ax.set(title=encoder_name.value, xlabel="Semantic boundary edges у labeled set", ylim=(0, 100))
        axes[0].set_ylabel("Val mIoU, %")
        return fig


def plot_class_distribution(distribution: pd.DataFrame, theme: str = CHART_THEME):
    cmap = LinearSegmentedColormap.from_list("class_frequency", ["#cde2fb", "#2a78d6", "#0d366b"])
    with plt.rc_context(chart_style(theme)):
        fig, axes = plt.subplots(2, 1, figsize=(14, 7), layout="constrained")
        for ax, encoder_name in zip(axes, BENCHMARK_ENCODERS):
            data = distribution.loc[
                (distribution.encoder == encoder_name.value)
                & ((distribution.budget == BENCHMARK_BUDGETS[-1]) | (distribution.method == "pool"))
            ]
            matrix = data.pivot(index="method", columns="class", values="image_presence_pct")
            matrix = matrix.reindex(index=["pool", *BENCHMARK_METHODS], columns=VOC_CLASSES)
            image = ax.imshow(matrix.to_numpy(), vmin=0, vmax=100, cmap=cmap, aspect="auto")
            ax.set(xticks=range(NUM_CLASSES), yticks=range(5), title=encoder_name.value)
            ax.set_xticklabels(VOC_CLASSES, rotation=45, ha="right")
            ax.set_yticklabels(["Pool", *[METHOD_LABELS[m] for m in BENCHMARK_METHODS]])
        fig.colorbar(image, ax=axes, label="Зображення з цим класом, %", shrink=0.8)
        return fig

In [30]:
def plot_first_batch(
    voc_root: Path, pool_ids: list[str], ids: list[int], title: str, theme: str = CHART_THEME,
):
    with plt.rc_context(chart_style(theme)):
        fig, axes = plt.subplots(math.ceil(len(ids) / 7), 7, figsize=(14, 7), squeeze=False)
        for ax in axes.flat:
            ax.axis("off")
        for ax, image_id in zip(axes.flat, ids):
            with Image.open(voc_root / "JPEGImages" / f"{pool_ids[image_id]}.jpg") as image:
                preview = image.convert("RGB")
                preview.thumbnail((180, 150))
            ax.imshow(preview)
            ax.set_title(f"ID {image_id}", fontsize=9)
        fig.suptitle(title)
        fig.tight_layout()
        return fig


def plot_uncertainty_snapshot(
    path: Path, voc_root: Path, pool_ids: list[str], title: str, theme: str = CHART_THEME,
):
    """Лише saved predictions, scores та RGB. Без inference й без GT masks."""
    with np.load(path, allow_pickle=False) as saved:
        ids, scores = saved["ids"], saved["scores"]
        predictions, uncertainty = saved["predictions"], saved["uncertainty"]
        most, temperature = saved["most_uncertain"], float(saved["temperature"])
    cmap = LinearSegmentedColormap.from_list("uncertainty", ["#cde2fb", "#2a78d6", "#0d366b"])
    edges_cmap = LinearSegmentedColormap.from_list("prediction_edges", ["#eb6834", "#eb6834"])
    with plt.rc_context(chart_style(theme)):
        fig, axes = plt.subplots(len(ids), 3, figsize=(10, 2.8 * len(ids)),
                                 squeeze=False, layout="constrained")
        for row, image_id in enumerate(ids):
            height, width = predictions[row].shape
            with Image.open(voc_root / "JPEGImages" / f"{pool_ids[image_id]}.jpg") as image:
                rgb = image.convert("RGB").resize((width, height), Image.Resampling.BICUBIC)
            boundary = np.zeros((height, width), dtype=bool)
            boundary[:, 1:] |= predictions[row, :, 1:] != predictions[row, :, :-1]
            boundary[1:] |= predictions[row, 1:] != predictions[row, :-1]
            axes[row, 0].imshow(rgb)
            axes[row, 1].imshow(rgb)
            axes[row, 1].imshow(np.ma.masked_where(~boundary, np.ones_like(boundary)),
                                cmap=edges_cmap, vmin=0, vmax=1, interpolation="nearest")
            heatmap = axes[row, 2].imshow(uncertainty[row], cmap=cmap, vmin=0, vmax=1)
            rank = "Найбільша" if most[row] else "Найменша"
            axes[row, 0].set_title(f"{rank} uncertainty, ID {image_id}")
            axes[row, 1].set_title("Межі передбачених класів")
            axes[row, 2].set_title(f"Image score {scores[row]:.3f}")
            for ax in axes[row]:
                ax.axis("off")
        fig.colorbar(heatmap, ax=list(axes[:, 2]), label="Pixel uncertainty", shrink=0.65)
        fig.suptitle(f"{title}, acquisition head, tau={temperature:g}")
        return fig

In [31]:
if RUN_POSTHOC:
    if benchmark_results.empty or not (benchmark_status.status == "complete").all():
        raise RuntimeError("Post-hoc labels читаємо лише після завершення всієї сітки")
    cost_table, class_table = posthoc_tables(benchmark_results, VOC_ROOT)
    cost_table.to_csv(report_dir / "annotation-cost.csv", index=False)
    class_table.to_csv(report_dir / "class-distribution.csv", index=False)
    display(cost_table[["encoder", "method", "budget", "boundary_edges", "miou"]].round(4))
    final_distribution = class_table.loc[
        (class_table.budget == BENCHMARK_BUDGETS[-1]) | (class_table.method == "pool")
    ].pivot(index=["encoder", "method"], columns="class", values="image_presence_pct")
    display(final_distribution.round(1))
    for name, figure in (
        ("annotation-cost", plot_annotation_cost(cost_table)),
        ("class-distribution", plot_class_distribution(class_table)),
    ):
        figure.savefig(report_dir / f"{name}.png", dpi=150)
        plt.show()
        plt.close(figure)
    for run_dir, config, method in benchmark_runs(BENCHMARK_ROOT):
        if method == "full_data":
            continue
        manifest = json.loads((run_dir / "manifest.json").read_text())
        selections = json.loads((run_dir / "selections.json").read_text())
        name = f"{config.encoder.name.value}-{method}"
        figure = plot_first_batch(
            VOC_ROOT, manifest["pool_ids"], selections[0]["new_ids"],
            f"{name}: перший queried batch",
        )
        figure.savefig(report_dir / f"first-batch-{name}.png", dpi=120)
        plt.show()
        plt.close(figure)
        snapshots = sorted((run_dir / "diagnostics").glob("round-*.npz"))
        if snapshots:
            figure = plot_uncertainty_snapshot(
                snapshots[-1], VOC_ROOT, manifest["pool_ids"], f"{name}, {snapshots[-1].stem}",
            )
            figure.savefig(report_dir / f"uncertainty-{name}.png", dpi=120)
            plt.show()
            plt.close(figure)
        elif method in ("uncertainty", "uherding"):
            print(f"{name}: знімків немає, їх не підмінено іншою моделлю")
else:
    print("Post-hoc читання train masks вимкнене. RUN_POSTHOC=False.")

Post-hoc читання train masks вимкнене. RUN_POSTHOC=False.


### Висновки після реальних експериментів

Результатів benchmark поки немає. Наведені раніше числові приклади синтетичні. Не робимо висновків про перевагу методу до завершення погодженої сітки.

Після запусків заповнюємо висновки за таблицями та графіками:

- Який метод має найкращий mIoU в low, mid і high budgets? Чи змінюється порядок методів між раундами?
- На яких бюджетах UHerding не гірший за обидва MaxHerding та Uncertainty top-B? Де `uherding_vs_better_baseline_pp` від'ємний?
- Як різниця між DINO та iBOT співвідноситься з gain over Random? Чи близькі методи до full-data upper bound?
- Чи зберігаються висновки на осі semantic boundary cost? Чим proxy відрізняється від реальної вартості розмітки?
- Що видно на uncertainty maps і в першому batch? Відокремлюємо спостереження від припущень про epistemic та aleatoric uncertainty.
- Який метод обрали б для заданого бюджету та чи виправдовує його gain додатковий час acquisition?

У benchmark до перегляду official val фіксуємо margin uncertainty. Entropy залишається реалізованим альтернативним score. Обидва encoders мають однаковий розмір ViT-B/16; відомості про ImageNet-1K pretraining і відсутність VOC fine-tuning беремо з зафіксованого source та metadata. Це не доводить відсутності перетину з VOC на рівні зображень.

### Перевірки та межі готовності

Детальні тести залишаються в `tests/`: `python -m unittest discover -s tests -v`. Вони перевіряють алгоритми, callback і report calculations на малих синтетичних fixtures. У notebook немає тестового benchmark чи вигаданих learning curves. Restart & Run All із трьома прапорцями `False` не запускає training, не завантажує weights і не читає GT masks для post-hoc аналізу.

Тести source contracts і cache лежать у `src/source/tests/`. Перевірки з реальним training або weights запускаємо лише з окремого дозволу. Full-data training, витрати GPU, працездатність реального VOC benchmark та resume не перевірені. Перед ними потрібні залежності, VOC masks, повний feature cache, persistent output directory та один погоджений короткий реальний запуск.